In [1]:
import os
import pickle
from collections import defaultdict

import clip
import numpy as np
import torch
from langchain.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.embeddings import Embeddings
from PIL import Image

In [ ]:
device = "cpu"
model, preprocess = clip.load("ViT-B/16", device=device)

In [ ]:
with open("data/embeddings.pkl", "rb") as f:
    embeddings_dict = pickle.load(f)

In [ ]:
class CLIPImageEmbeddings(Embeddings):
    def __init__(self, device):
        self.device = device
        self.model, self.preprocess = clip.load("ViT-B/16", device=self.device)

    def embed_documents(self, texts):
        # Not needed for this use case since we are only doing query embedding
        raise NotImplementedError

    def embed_query(self, image_path):
        # Use the provided `vectorize_img` function
        return self.vectorize_img(image_path)

    def vectorize_img(self, img_path):
        # Preprocess image and get embedding
        img = self.preprocess(Image.open(img_path)).unsqueeze(0).to(self.device)
        with torch.no_grad():
            embedding = self.model.encode_image(img)
        embedding /= np.linalg.norm(embedding)
        return embedding.cpu().numpy()


embedder = CLIPImageEmbeddings(device=device)

embeddings = np.array(list(embeddings_dict.values())).squeeze()
file_names = list(embeddings_dict.keys())

norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
normalized_embeddings = embeddings / norms


# Create the vector store with the NORMALIZED embeddings
vectorstore = FAISS.from_embeddings(
    text_embeddings=list(
        zip(file_names, normalized_embeddings)
    ),  # Use the new variable
    embedding=embedder,
    distance_strategy=DistanceStrategy.COSINE,
)

In [ ]:
user_query = "data/test_img_1.jpg"
query_vector = embedder.embed_query(user_query)
results = vectorstore.similarity_search_with_score_by_vector(
    query_vector.flatten(), k=5
)

In [ ]:
results

In [ ]:
def find_best_class_id(results):
    """
    Analyzes search results to find the class with the lowest cumulative distance.

    This version assumes every filename is correctly formatted as 'OXXXX_YYYYYY.jpg',
    where the class ID is the numeric part 'XXXX'.

    Args:
        results (list): A list of tuples, where each tuple contains a
                        langchain Document object and a distance score.

    Returns:
        int: The integer ID of the class with the least cumulative distance.
             Returns None if the input list is empty.
    """
    class_scores = defaultdict(float)

    for doc, score in results:
        # Directly parse the class ID from the guaranteed filename format
        class_id_str = doc.page_content.split("_")[0][1:]
        class_id = int(class_id_str)

        # Add the score to the cumulative total for that class ID
        class_scores[class_id] += score

    # Find and return the class ID with the minimum cumulative score
    best_id = min(class_scores, key=class_scores.get)
    return best_id

In [ ]:
final_class = find_best_class_id(results)

In [ ]:
final_class

In [ ]:
vectorstore.save_local(folder_path="artifacts")

### Test

In [3]:
from rag_searcher import RAGSearcher

ragger = RAGSearcher(
    device="cpu",
    vectorstore_path="artifacts/db",
    object_descr_path="artifacts/metro_objects.json",
)

In [4]:
def check_accuracy(ragger, images, path):
    s_true, s_done = 0, 0

    for key, val in images.items():
        for image in val[:-3]:
            output = ragger.search(f"{path}{image[:5]}/{image}")
            s_done += 1
            if output == int(key[1:]):
                s_true += 1

    return s_true / s_done


base = {}
path = "data/images/"
obj = os.listdir(path)

for folder in obj:
    base[folder] = sorted(os.listdir(f"data/images/{folder}"))


print(check_accuracy(ragger, base, path))

0.9568965517241379
